In [ ]:
#IMPORT STATEMENTS
import re
from typing import Dict, List, Tuple
from collections import Counter


Processing Amazon Product Reviews...

AMAZON PRODUCT REVIEW ANALYSIS

Review #1
--------------------------------------------------------------------------------
Text: I absolutely love my new Apple iPhone! The camera quality is amazing and the battery life is excellent. Best phone I've ever owned.

Extracted Entities:
  Brands: Apple
  Products: Best phone, The camera

Sentiment Analysis:
  Overall: POSITIVE
  Confidence: 1.0
  Positive Score: 5.5
  Negative Score: 0


Review #2
--------------------------------------------------------------------------------
Text: Very disappointed with these Samsung earbuds. The sound quality is poor and they're uncomfortable to wear. Not worth the money.

Extracted Entities:
  Brands: Samsung
  Products: Samsung earbuds

Sentiment Analysis:
  Overall: NEGATIVE
  Confidence: 0.82
  Positive Score: 1.0
  Negative Score: 4.5


Review #3
--------------------------------------------------------------------------------
Text: This Amazon Kindle is fantasti

In [ ]:
#NAMED ENTITY RECOGNITION FOR PRODUCTS AND BRANDS
class ProductNER:
    """Extract product names and brands from reviews using pattern matching"""

    def __init__(self):
        # Common brand keywords and patterns
        self.brand_patterns = [
            r'\b(Amazon|Apple|Samsung|Sony|LG|Dell|HP|Lenovo|Microsoft|Google|Nike|Adidas|Puma)\b',
            r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\s+(?:brand|Brand)\b',
        ]

        # Product type keywords
        self.product_keywords = [
            'phone', 'laptop', 'tablet', 'camera', 'headphones', 'earbuds',
            'speaker', 'watch', 'charger', 'cable', 'case', 'keyboard',
            'mouse', 'monitor', 'TV', 'book', 'shoes', 'shirt', 'pants',
            'dress', 'jacket', 'bag', 'backpack', 'bottle', 'mug'
        ]

    def extract_brands(self, text: str) -> List[str]:
        """Extract brand names from text"""
        brands = []
        for pattern in self.brand_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            brands.extend([m if isinstance(m, str) else m[0] for m in matches])
        return list(set(brands))

    def extract_products(self, text: str) -> List[str]:
        """Extract product mentions from text"""
        products = []
        text_lower = text.lower()

        for keyword in self.product_keywords:
            if keyword.lower() in text_lower:
                # Extract context around the keyword
                pattern = rf'\b\w+\s+{keyword}\b|\b{keyword}\b'
                matches = re.findall(pattern, text, re.IGNORECASE)
                products.extend(matches)

        return list(set(products))

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract all entities from text"""
        return {
            'brands': self.extract_brands(text),
            'products': self.extract_products(text)
        }

In [ ]:
#SENTIMENT ANALYSIS USING RULE-BASED APPROACH
class SentimentAnalyzer:
    """Rule-based sentiment analysis using lexicon approach"""

    def __init__(self):
        # Positive sentiment words
        self.positive_words = {
            'excellent', 'amazing', 'great', 'good', 'wonderful', 'fantastic',
            'awesome', 'best', 'perfect', 'love', 'loved', 'beautiful',
            'brilliant', 'outstanding', 'superb', 'impressive', 'quality',
            'recommend', 'happy', 'satisfied', 'worth', 'reliable', 'durable',
            'comfortable', 'fast', 'easy', 'nice', 'pleased', 'favorite'
        }

        # Negative sentiment words
        self.negative_words = {
            'terrible', 'awful', 'bad', 'horrible', 'worst', 'poor', 'disappointed',
            'disappointing', 'waste', 'useless', 'broken', 'defective', 'cheap',
            'uncomfortable', 'slow', 'difficult', 'hate', 'hated', 'regret',
            'unreliable', 'flimsy', 'garbage', 'junk', 'avoid', 'never', 'not worth'
        }

        # Intensifiers
        self.intensifiers = {'very', 'really', 'extremely', 'absolutely', 'completely'}

        # Negations
        self.negations = {'not', 'no', 'never', 'nothing', 'neither', 'nobody', "n't"}

    def analyze_sentiment(self, text: str) -> Dict[str, any]:
        """Analyze sentiment of text using rule-based approach"""
        words = re.findall(r'\b\w+\b', text.lower())

        positive_score = 0
        negative_score = 0

        for i, word in enumerate(words):
            # Check for negation in previous 3 words
            negated = False
            if i > 0:
                prev_words = words[max(0, i-3):i]
                negated = any(neg in prev_words for neg in self.negations)

            # Check for intensifier
            intensifier = 1.0
            if i > 0 and words[i-1] in self.intensifiers:
                intensifier = 1.5

            if word in self.positive_words:
                if negated:
                    negative_score += 1 * intensifier
                else:
                    positive_score += 1 * intensifier

            elif word in self.negative_words:
                if negated:
                    positive_score += 1 * intensifier
                else:
                    negative_score += 1 * intensifier

        # Calculate overall sentiment
        total = positive_score + negative_score
        if total == 0:
            sentiment = 'neutral'
            confidence = 0.0
        else:
            if positive_score > negative_score:
                sentiment = 'positive'
                confidence = positive_score / total
            elif negative_score > positive_score:
                sentiment = 'negative'
                confidence = negative_score / total
            else:
                sentiment = 'neutral'
                confidence = 0.5

        return {
            'sentiment': sentiment,
            'confidence': round(confidence, 2),
            'positive_score': positive_score,
            'negative_score': negative_score
        }

In [ ]:
#MAIN ANALYSIS FUNCTION
def analyze_reviews(reviews: List[str]) -> List[Dict]:
    """Analyze a list of reviews for entities and sentiment"""
    ner = ProductNER()
    sentiment_analyzer = SentimentAnalyzer()

    results = []
    for idx, review in enumerate(reviews, 1):
        entities = ner.extract_entities(review)
        sentiment = sentiment_analyzer.analyze_sentiment(review)

        results.append({
            'review_id': idx,
            'review': review,
            'entities': entities,
            'sentiment': sentiment
        })

    return results

In [ ]:
#PRINTING AND SUMMARY FUNCTIONS
def print_analysis_results(results: List[Dict]):
    """Pretty print analysis results"""
    print("=" * 80)
    print("AMAZON PRODUCT REVIEW ANALYSIS")
    print("=" * 80)
    print()

    for result in results:
        print(f"Review #{result['review_id']}")
        print("-" * 80)
        print(f"Text: {result['review']}")
        print()
        print(f"Extracted Entities:")
        print(f"  Brands: {', '.join(result['entities']['brands']) or 'None detected'}")
        print(f"  Products: {', '.join(result['entities']['products']) or 'None detected'}")
        print()
        print(f"Sentiment Analysis:")
        print(f"  Overall: {result['sentiment']['sentiment'].upper()}")
        print(f"  Confidence: {result['sentiment']['confidence']}")
        print(f"  Positive Score: {result['sentiment']['positive_score']}")
        print(f"  Negative Score: {result['sentiment']['negative_score']}")
        print()
        print("=" * 80)
        print()

In [ ]:
#SUMMARY STATISTICS FUNCTION
def generate_summary(results: List[Dict]):
    """Generate summary statistics"""
    total_reviews = len(results)
    sentiments = [r['sentiment']['sentiment'] for r in results]
    sentiment_counts = Counter(sentiments)

    all_brands = []
    all_products = []
    for r in results:
        all_brands.extend(r['entities']['brands'])
        all_products.extend(r['entities']['products'])

    brand_counts = Counter(all_brands)
    product_counts = Counter(all_products)

    print("SUMMARY STATISTICS")
    print("=" * 80)
    print(f"Total Reviews Analyzed: {total_reviews}")
    print()
    print("Sentiment Distribution:")
    for sentiment, count in sentiment_counts.items():
        percentage = (count / total_reviews) * 100
        print(f"  {sentiment.capitalize()}: {count} ({percentage:.1f}%)")
    print()
    print("Most Mentioned Brands:")
    for brand, count in brand_counts.most_common(5):
        print(f"  {brand}: {count} mentions")
    print()
    print("Most Mentioned Products:")
    for product, count in product_counts.most_common(5):
        print(f"  {product}: {count} mentions")
    print("=" * 80)

In [ ]:
#MAIN EXECUTION BLOCK
if __name__ == "__main__":
    # Sample Amazon product reviews
    sample_reviews = [
        "I absolutely love my new Apple iPhone! The camera quality is amazing and the battery life is excellent. Best phone I've ever owned.",

        "Very disappointed with these Samsung earbuds. The sound quality is poor and they're uncomfortable to wear. Not worth the money.",

        "This Amazon Kindle is fantastic! Easy to use, great screen, and the battery lasts forever. Highly recommend for book lovers.",

        "The Nike running shoes are good but not great. Comfortable for short runs but lack support for longer distances.",

        "Terrible laptop! The Dell keyboard broke after just 2 weeks and customer service was awful. Complete waste of money.",

        "Really happy with my Sony headphones. Excellent noise cancellation and the sound quality is superb. Worth every penny!",

        "This phone case is cheap and flimsy. Broke within days. Do not buy!",

        "The Google smart speaker works perfectly. Easy setup, responds quickly, and the sound is really impressive for the size.",
    ]

    print("\nProcessing Amazon Product Reviews...\n")

    # Analyze all reviews
    results = analyze_reviews(sample_reviews)

    # Print detailed results
    print_analysis_results(results)

    # Print summary
    generate_summary(results)